# XGBoost Regression on the Full ESPN Dataset: Next-Game Points

This notebook trains an **XGBoost** model to predict a player's points in a game, using the full feature table that `scripts/build_dataset.py` writes to `data/NBA_player_stats.csv` (2021-22 through 2025-26 ESPN box scores).

**Target:** `target_pts`, the points scored in that row's game.

**Leakage rule:** `data/processing/full_features.py` builds every feature from games *before* the row's game (sort by date, shift one game, then roll). Same-game box-score values appear only in `target_*` columns, and none of those are used as features here. The feature list comes from `full_features.feature_columns`, so it updates automatically when the builder adds a feature.

**Split:** the dataset's own season split.

| split | seasons | used for |
|---|---|---|
| train | 2021-22, 2022-23, 2023-24 | fitting |
| val | 2024-25 | hyperparameter tuning and early stopping |
| test | 2025-26 | one final evaluation |

The final model is refit on train + val, then scored once on test.

`ml/training/XGBoost.ipynb` is the earlier, 5-feature version on the sample CSV. This notebook does not replace it.

## Config

- `POINTS_DATA_PATH` (environment variable): path to a different feature CSV. If it is not set, the notebook uses `data/NBA_player_stats.csv`.
- `PARAM_GRID`: the hyperparameters tried. Each combination is fit on train and early-stopped on val.
- `MAX_ESTIMATORS` / `EARLY_STOPPING_ROUNDS`: boosting stops once val MAE has not improved for this many rounds.
- `SAVE_MODEL`: off by default, so running the notebook does not write artifacts or touch `ml/models/registry.json`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor

# Make the repo root importable so shared project code can be used here.
REPO_ROOT = next(
    (root for root in (Path.cwd(), *Path.cwd().parents) if (root / 'data' / 'processing' / 'full_features.py').exists()),
    None,
)
assert REPO_ROOT is not None, 'Could not find the repo root (data/processing/full_features.py)'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.processing import full_features
from ml.training.evaluation import regression_metrics

DATA_PATH = Path(os.environ.get('POINTS_DATA_PATH', REPO_ROOT / 'data' / 'NBA_player_stats.csv'))
if not DATA_PATH.exists():
    raise FileNotFoundError(f'{DATA_PATH} not found. Build it first with: python -m scripts.build_dataset')

TARGET = 'target_pts'
BASELINES = {
    'Season avg': 'pts_season_avg',
    'Last 5 avg': 'pts_last5_avg',
    'Last 10 avg': 'pts_last10_avg',
    'EWM': 'pts_ewm',
}

RANDOM_STATE = 42
MAX_ESTIMATORS = 2000
EARLY_STOPPING_ROUNDS = 100
PARAM_GRID = {
    'max_depth': [4, 6],
    'learning_rate': [0.03, 0.1],
    'min_child_weight': [5, 20],
    'subsample': [0.8],
    'colsample_bytree': [0.7, 1.0],
}

SAVE_MODEL = False
MODEL_NAME = 'xgboost-points-full'

print(DATA_PATH)
print('xgboost', xgboost.__version__)

## Load the dataset and prepare features

- `FEATURES` is every column that is not an identity column and not a `target_*` column.
- Boolean features become 0/1. `position` and `pos_group` become pandas categories, which XGBoost splits on directly (`enable_categorical=True`). The categories are set on the whole table, so every split uses the same codes.
- Missing values stay as NaN. They mostly mean the player has no history yet, and XGBoost learns which side of each split to send them to, so no imputer is needed.

In [ ]:
df = pd.read_csv(DATA_PATH, dtype={'game_id': str})
df['game_date'] = pd.to_datetime(df['game_date'], format='%Y-%m-%d')

FEATURES = full_features.feature_columns(df.columns)
assert not any(c.startswith('target_') for c in FEATURES), 'A same-game target leaked into FEATURES'
assert set(df['split'].dropna()) <= {'train', 'val', 'test'}, df['split'].unique()
assert df[TARGET].notna().all(), 'Rows without a target'

BOOL_STRINGS = {True: 1, False: 0, 'True': 1, 'False': 0}
CATEGORICAL = []
for column in FEATURES:
    values = df[column]
    if values.dtype == bool:
        df[column] = values.astype(int)
    elif pd.api.types.is_object_dtype(values) or pd.api.types.is_string_dtype(values):
        if set(values.dropna().unique()) <= set(BOOL_STRINGS):
            df[column] = values.map(BOOL_STRINGS).astype(float)
        else:
            df[column] = values.astype('category')
            CATEGORICAL.append(column)

train = df[df['split'] == 'train']
val = df[df['split'] == 'val']
test = df[df['split'] == 'test']
train_val = df[df['split'].isin(['train', 'val'])]

print(f'{len(FEATURES)} features; categorical: {CATEGORICAL}')
pd.DataFrame({
    name: {
        'rows': len(frame),
        'players': frame['player_id'].nunique(),
        'first_game': frame['game_date'].min().date(),
        'last_game': frame['game_date'].max().date(),
        'mean_pts': round(frame[TARGET].mean(), 2),
    }
    for name, frame in [('train', train), ('val', val), ('test', test)]
}).T

In [ ]:
# Share of missing values per feature (only features with any NaN).
nulls = (df[FEATURES].isna().mean() * 100).round(2)
nulls[nulls > 0].sort_values(ascending=False).to_frame('null %')

## Baselines

Each baseline predicts the target with one prior-only feature. They are scored on the rows where **every** baseline has a value (`common` rows), so all of them see the same games. The model is later scored on these same rows as well.

In [ ]:
def common_rows(frame):
    return frame.dropna(subset=list(BASELINES.values()))


def score_rows(frame, predictions):
    """Baselines plus any named prediction Series, all on exactly `frame`'s rows."""
    scores = {name: regression_metrics(frame[TARGET], frame[column]) for name, column in BASELINES.items()}
    for name, pred in predictions.items():
        scores[name] = regression_metrics(frame[TARGET], pred.loc[frame.index])
    return pd.DataFrame(scores).T


val_common, test_common = common_rows(val), common_rows(test)
print(f'val: {len(val_common):,}/{len(val):,} common rows; test: {len(test_common):,}/{len(test):,}')
score_rows(val_common, {}).sort_values('MAE')

## Hyperparameter tuning on val

Each combination in `PARAM_GRID` is fit on train, with up to `MAX_ESTIMATORS` trees, and early-stopped on val MAE. The best iteration is recorded so the final model can use a matching number of trees. Early stopping looks at the same val rows it is scored on, so these val scores are slightly optimistic. Test is not used here.

- `max_depth`: how deep each tree can grow.
- `learning_rate`: how much each new tree contributes.
- `min_child_weight`: the minimum amount of data in a leaf. Larger values make trees more conservative.
- `subsample` / `colsample_bytree`: the share of rows and features sampled for each tree, which adds randomness and reduces overfitting.

In [ ]:
def make_xgb(**params):
    return XGBRegressor(
        objective='reg:squarederror',
        tree_method='hist',
        enable_categorical=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params,
    )


X_train, y_train = train[FEATURES], train[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]

grid = list(ParameterGrid(PARAM_GRID))
tuning_rows = []
for i, params in enumerate(grid):
    candidate = make_xgb(n_estimators=MAX_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                         eval_metric='mae', **params)
    candidate.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    pred = candidate.predict(X_val)
    tuning_rows.append({'candidate': i, **params, 'best_n_trees': candidate.best_iteration + 1,
                        **regression_metrics(y_val, pred)})
    print(f'[{i + 1}/{len(grid)}] {params} -> val MAE {tuning_rows[-1]["MAE"]:.3f}')

tuning = pd.DataFrame(tuning_rows).sort_values('MAE').reset_index(drop=True)
best_params = grid[int(tuning.loc[0, 'candidate'])]
best_n_trees = int(tuning.loc[0, 'best_n_trees'])
print(f'Best: {best_params}, {best_n_trees} trees, val MAE {tuning.loc[0, "MAE"]:.3f}')
tuning.drop(columns='candidate').head(10)

## Train the final model on train + val

The final model is refit on train + val with the best parameters. There is no early-stopping set left, so the tree count is the best val iteration scaled up by how much more data the refit has. This is a common rule of thumb, not a tuned value.

In [ ]:
final_n_estimators = max(1, round(best_n_trees * len(train_val) / len(train)))
final_params = {**best_params, 'n_estimators': final_n_estimators}
X_train_val, y_train_val = train_val[FEATURES], train_val[TARGET]
model = make_xgb(**final_params).fit(X_train_val, y_train_val)
print(f'Final model: {final_params}')

## Evaluate on test (2025-26)

This is the only time test is used. The model and every baseline are scored on the same common rows. The breakdown then repeats the model's score on subsets:

- **all test rows:** includes players with no history, where the baselines are NaN.
- **≥ 5 prior games** (`insufficient_history == 0`)
- **rotation players** (`low_minutes_prior == 0`)

In [ ]:
test_pred = pd.Series(model.predict(test[FEATURES]), index=test.index)

test_scores = score_rows(test_common, {'XGBoost': test_pred})
best_baseline = test_scores.drop(index='XGBoost')['MAE'].idxmin()
beats = test_scores.loc['XGBoost', 'MAE'] < test_scores.loc[best_baseline, 'MAE']
print(f'Best baseline: {best_baseline} -> XGBoost beats it: {beats}')
test_scores.sort_values('MAE')

In [ ]:
subsets = {
    'all test rows': test,
    'common rows': test_common,
    '>= 5 prior games': test[test['insufficient_history'] == 0],
    'rotation players': test[test['low_minutes_prior'] == 0],
}
pd.DataFrame({name: regression_metrics(rows[TARGET], test_pred.loc[rows.index])
              for name, rows in subsets.items() if len(rows)}).T

## Diagnostics

- **Feature importance (gain):** how much each feature's splits reduced the loss on average. The top 25 are shown.
- **Predicted vs actual:** points on the diagonal are perfect predictions.
- **Error by expected minutes:** MAE grouped by `min_ewm`, the player's recent minutes.
- **Residuals by player:** players whose test games the model gets most wrong.

In [ ]:
importance = pd.Series(model.get_booster().get_score(importance_type='gain')).reindex(FEATURES).fillna(0)
importance = importance / importance.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
importance.sort_values().tail(25).plot.barh(ax=axes[0], color='tab:blue')
axes[0].set_title('XGBoost feature importance (share of gain, top 25)')
axes[0].set_xlabel('share of total gain')

lim = [0, max(test[TARGET].max(), test_pred.max()) + 2]
axes[1].scatter(test[TARGET], test_pred, s=4, alpha=0.2)
axes[1].plot(lim, lim, 'k--', linewidth=1)
axes[1].set(xlim=lim, ylim=lim, xlabel='actual points', ylabel='predicted points',
            title=f"Test: predicted vs actual (MAE {regression_metrics(test[TARGET], test_pred)['MAE']:.2f})")
plt.tight_layout()
plt.show()

In [ ]:
residuals = test.assign(predicted=test_pred, error=test_pred - test[TARGET],
                        abs_error=(test_pred - test[TARGET]).abs())
minutes_bucket = pd.cut(residuals['min_ewm'], [0, 10, 20, 30, 40, 60], right=False)
(residuals.groupby(minutes_bucket.cat.add_categories('no history').fillna('no history'), observed=True)
 .agg(rows=('error', 'size'), MAE=('abs_error', 'mean'), bias=('error', 'mean'), avg_points=(TARGET, 'mean'))
 .round(2))

In [ ]:
by_player = (residuals.groupby('player_name')
             .agg(games=('error', 'size'), MAE=('abs_error', 'mean'), bias=('error', 'mean'),
                  avg_points=(TARGET, 'mean'))
             .query('games >= 10')
             .sort_values('MAE', ascending=False)
             .round(2))
print('Players with the largest test error (bias > 0 means over-predicted):')
by_player.head(10)

## Save the model (optional)

When `SAVE_MODEL = True`, this calls `ml.model_integrity.save_model`, the same as `train_tree_points.py` and `XGBoost.ipynb`. It writes the artifact, its manifest (with the config and metrics above) and a `candidate` entry in `ml/models/registry.json`. It is off by default.

In [ ]:
if SAVE_MODEL:
    from ml.model_integrity import save_model, sha256_file

    training_config = {
        'training_script': 'ml/training/xgboost_full_dataset.ipynb',
        'feature_builder': 'data/processing/full_features.py',
        'estimator': 'XGBRegressor(objective=reg:squarederror, tree_method=hist, enable_categorical=True)',
        'xgboost_params': {**final_params, 'random_state': RANDOM_STATE},
        'tuning': {'param_grid': PARAM_GRID, 'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
                   'best_val_n_trees': best_n_trees, 'selected_on': 'val'},
        'features': FEATURES,
        'categorical_features': CATEGORICAL,
        'target': TARGET,
        'split': {'fit': 'train + val (2021-22 to 2024-25)', 'tuning': 'val (2024-25)', 'test': 'test (2025-26)'},
        'dataset': str(DATA_PATH.relative_to(REPO_ROOT)) if DATA_PATH.is_relative_to(REPO_ROOT) else str(DATA_PATH),
        'dataset_sha256': sha256_file(DATA_PATH),
        'xgboost_version': xgboost.__version__,
    }
    evaluation_metrics = {
        'model': 'XGBoost',
        'val_tuning': tuning.drop(columns='candidate').iloc[0].to_dict(),
        'test': test_scores.loc['XGBoost'].to_dict(),
        'baselines_test': test_scores.drop(index='XGBoost').to_dict(orient='index'),
        'best_baseline': best_baseline,
        'beats_best_baseline': bool(beats),
        'test_all_rows': regression_metrics(test[TARGET], test_pred),
    }
    entry = save_model(model, model_name=MODEL_NAME, features=X_train_val, dataset_path=DATA_PATH,
                       training_config=training_config, evaluation_metrics=evaluation_metrics)
    print(json.dumps({'saved': entry['model_version'], 'artifact': entry['artifact_path']}, indent=2))
else:
    print('SAVE_MODEL is False; nothing written.')